# **İşaretleyici tabanlı görüntü segmentasyonu için  Watershed Algoritması**

#### **Bu derste şunları öğreneceğiz:**
1. İşaretleyici tabanlı görüntü segmentasyonu için Watershed algoritması nasıl kullanılır


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


# **Watershed  Algoritması Teorisi**

Herhangi bir gri tonlamalı görüntü, yüksek yoğunluğun tepeleri, düşük yoğunluğun ise vadileri gösterdiği bir topografik yüzey olarak görülebilir.

Watershed algoritması bu benzetmeyi kullanır ve bu alçak noktaları (vadileri) farklı renkte bir etiketle (yani suyumuzla) doldurmaya başlar.

Su yükseldikçe, yakındaki tepelere (eğimlere) bağlı olarak, farklı vadilerden gelen ve açıkça farklı renklere sahip sular birleşmeye başlayacaktır. Bunu önlemek için suyun birleştiği yerlere bariyerler inşa edersiniz. Tüm tepeler su altında kalana kadar su doldurma ve bariyerler inşa etme işine devam edersiniz.

Oluşturduğunuz bariyerler size segmentasyon sonucunu verir. Bu, su havzasının arkasındaki "felsefedir". 

Ancak bu yaklaşım, gürültü veya görüntüdeki diğer düzensizlikler nedeniyle aşırı boyutlandırılmış sonuçlar verir.

Bu nedenle OpenCV, hangi vadi noktalarının birleştirileceğini ve hangilerinin birleştirilmeyeceğini belirlediğiniz işaretleyici tabanlı bir watershed algoritması uygular. Bildiğimiz nesnelerimiz için farklı etiketler verir. Ön plan veya nesne olduğundan emin olduğumuz bölgeyi bir renkle (veya yoğunlukla) etiketler, arka plan veya nesne olmadığından emin olduğumuz bölgeyi başka bir renkle etiketler ve son olarak hiçbir şeyden emin olmadığımız bölgeyi 0 ile etiketler. Daha sonra watershed algoritmasını uygulanır. Daha sonra işaretleyicimiz verdiğimiz etiketlerle güncellenecek ve nesnelerin sınırları -1 değerine sahip olacaktır.




In [ ]:
# Görüntüyü yükle
img = cv2.imread('../files/images/america31.jpg')
imshow("Original image", img,5)

# Gri Tonlama
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# OTSU kullanarak eşikle
ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

imshow("Thresholded", thresh,5)

## **Dokunan maskelerin çıkarılması**

In [ ]:
# gürültü giderme
kernel = np.ones((3,3), np.uint8) # Bir kernel tanımlıyoruz 3x3
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,kernel, iterations = 2) 
# Önce erosion (aşındırma), ardından dilation (genişletme) uygular.
imshow("opening", opening,5)
#arka plan alanını belirginleştir
sure_bg = cv2.dilate(opening, kernel, iterations=3) # beyaz bölgeler genişletilir.
imshow("SureBG", sure_bg,5)
# Kesin ön plan alanını bulma
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2,5)
ret, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)
# cv2.distanceTransform: Her pikselin en yakın sıfır (siyah) piksele olan mesafesini hesaplar.
#Thresholding ile mesafenin %70’inden daha büyük olan bölgeler kesin ön plan (sure foreground) kabul edilir.
#Bu, nesnelerin merkezlerini belirler.

# Bilinmeyen bölgeyi bulma
sure_fg = np.uint8(sure_fg)

imshow("SureFG", sure_fg,5)

unknown = cv2.subtract(sure_bg, sure_fg)
imshow("unknown", unknown,5)

In [ ]:
# İşaretleyici etiketleme
# Bağlı Bileşenler, ikili bir görüntüdeki blob benzeri bölgelerin bağlanabilirliğini belirler.
ret, markers = cv2.connectedComponents(sure_fg)

# Arka planın 0 değil 1 olduğundan emin olmak için tüm etiketlere bir ekleyin
markers = markers+1

# Şimdi, bilinmeyen bölgeyi sıfır ile işaretleyelim
markers[unknown==255] = 0

markers = cv2.watershed(img,markers)
img[markers == -1] = [255,0,0]

imshow("img", img,5)